# Agents와 Sturctured Outputs

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """영화에 대한 세부 정보."""
    title: str = Field(..., description="영화의 제목")
    year: int = Field(..., description="영화의 개봉된 연도")
    director: str = Field(..., description="영화의 감독")
    rating: float = Field(..., description="10점 만점 영화 평점")

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[],
    response_format=Movie
)

In [4]:
agent.invoke(
    {"messages": [{"role": "user", "content": "영화 기생충에 대한 세부 정보를 제공해줘"}]},
)

{'messages': [HumanMessage(content='영화 기생충에 대한 세부 정보를 제공해줘', additional_kwargs={}, response_metadata={}, id='98aa5660-288c-46a9-a798-ea90020352ec'),
  AIMessage(content='{"title":"기생충","year":2019,"director":"봉준호","rating":8.5}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019bac43-5e3e-79b1-919c-b6c8dfa9faf5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 156, 'total_tokens': 169, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 130}})],
 'structured_response': Movie(title='기생충', year=2019, director='봉준호', rating=8.5)}

### 도구 호출 전략

In [1]:
# 테스트용 모의 데이터
mock_emails = {
    101: {
        "sender": "angry_customer@gmail.com",
        "subject": "배송 지연 문의",
        "body": "안녕하세요. 지난주에 주문한 노트북 배송이 계속 지연되고 있습니다. 벌써 3일째인데 아무런 연락이 없네요. 매우 실망스럽습니다. 빠른 확인 바랍니다."
    },
    102: {
        "sender": "happy_user@naver.com",
        "subject": "업데이트 칭찬",
        "body": "새로 업데이트된 기능을 써봤는데 정말 편리하네요! 특히 다크 모드가 눈이 안 아파서 좋습니다. 개발팀에 감사드려요."
    },
    103: {
        "sender": "wondering@kakao.com",
        "subject": "환불 규정 문의",
        "body": "환불 규정이 궁금합니다. 단순 변심인 경우에도 배송비 무료인가요?"
    }
}

In [2]:
from langchain.tools import tool
import json

@tool
def read_email(email_id: int) -> str:
    """ID를 통해 이메일 상세 내용(발신자, 제목, 본문)을 읽어옵니다."""
    email_data = mock_emails.get(email_id)

    if not email_data:
        return "해당 ID의 이메일을 찾을 수 없습니다."
    
    print(f"[시스템] 이메일 ID {email_id} 읽기 완료 (발신자: {email_data["sender"]})")

    return json.dumps(email_data, ensure_ascii=False)

@tool
def send_email(to: str, subject: str, body: str) -> bool:
    """고객에게 이메일을 직접 발송합니다."""
    print(f"\n이메일 발송 중...")
    print(f"받는 사람: {to}")
    print(f"제목: {subject}")
    print(f"내용: {body}")
    print(f"-" * 30)

    return True

In [3]:
from pydantic import BaseModel, Field
from typing import List, Literal

class CustomerServiceReport(BaseModel):
    intent: str = Field(description="고객의 핵심 의도 (예: 배송 불만, 기능 칭찬, 규정 문의 등)")
    sentiment: Literal["positive", "neutral", "negative"] = Field(description="고객의 감성 상태 분석")
    summary: str = Field(description="이메일 내용의 1줄 요약")
    required_actions: List[str] = Field(description="이 문제를 해결하기 위해 내부에서 처리해야 할 조치 리스트")
    composed_reply: str = Field(description="고객의 감성을 케어하고 상황을 해결하는 정중한 답장 본문")

In [7]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[read_email, send_email],
    response_format=ToolStrategy(CustomerServiceReport)
)

In [8]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "103번 이메일을 분석하고, 분석 내용에 기반해 해당 고객(발신자)에게 적절한 답장을 직접 보내줘."
        }
    ]
})

response

[시스템] 이메일 ID 103 읽기 완료 (발신자: wondering@kakao.com)

이메일 발송 중...
받는 사람: wondering@kakao.com
제목: 환불 규정 문의 답변
내용: 안녕하세요, 고객님. 환불 규정에 대해 문의주셔서 감사합니다. 단순 변심으로 인한 환불의 경우, 반품 배송비는 고객님께서 부담하시게 됩니다. 더 자세한 내용은 저희 웹사이트의 환불 정책 페이지를 참고해주시거나, 추가 문의사항이 있으시면 언제든지 다시 연락주시기 바랍니다. 감사합니다.
------------------------------


{'messages': [HumanMessage(content='103번 이메일을 분석하고, 분석 내용에 기반해 해당 고객(발신자)에게 적절한 답장을 직접 보내줘.', additional_kwargs={}, response_metadata={}, id='3d141ba4-53a2-4cd8-b42c-5b2d905dc7b2'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{"email_id": 103}'}, '__gemini_function_call_thought_signatures__': {'b52144e9-586d-4420-90b8-133b84326e31': 'Cq0EAXLI2nyui8gIt37ls1xzJOHXhF4zqKL1Lsk0nhmGOpx37WDmxy9e/sI03gWd8wCOPmWudYoIynbwjTaAM/kwJgUodrcMDYO6vfsoqcY3VCSdfNOja1jKdwxWk7sMyn478FQAnYG4hx6ObQ8WLbNQRR30qA7VyntiR8g1rAT8feBDEaKSoH5FGQhO5TAZx4VbkjyTx9+sMoZ8YaUYgvtr7ExnWpVYjAsNNasjXDIviRlikIHS1BkwISA6/6ozHwKQuMB9Yaz6KE0yAVcbzdITPN1Z32CH7shpedlhicBdxMa4V9EOmaozUiLFoMnyr7VZ0+00QE5dKSFvjoNtYXMkNdJSFIVIFpbz4M/WRzL5aEpq6BQD4TJPuih0tHrEX5dMqqfh++d0pcnjNlB4ZLv5mOpvj/a9CAW2/97O+68yuZGQaFoCzuRDV25dE62e+eo247Bxu8XXAHQA78D7IPkczvUt7Ef8wWYDHjORXyt0OdBuimE0k54tukqr4Ps0Bh370qY/kt+h5xucsYbzCIqUMEDyUZmN63iAtkQj+SYYSj5N/31Oja4n3ACS4Thj7ZnqkwZXvwB1BfZngjenAGk4dWlVx4gjIcp